In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import acorr_ljungbox

panel = pd.read_parquet("Panel.parquet")

HAR  = ["RV_daily", "RV_weekly", "RV_monthly"]
HARX = HAR + ["CDS_level", "CDS_change_5d", "VIX", "Yield_slope", "BAA10Y"]
d = panel.dropna(subset=HARX + ["Target_h1"]).copy()

print(f"Diagnostic sample: {len(d):,} firm-day observations")

Diagnostic sample: 202,463 firm-day observations


In [2]:
corr = d[HARX].corr().round(3)
corr.to_csv("Correlation_Matrix.csv")
print("Correlation matrix of model variables:\n")
print(corr.to_string())

Correlation matrix of model variables:

               RV_daily  RV_weekly  RV_monthly  CDS_level  CDS_change_5d    VIX  Yield_slope  BAA10Y
RV_daily          1.000      0.564       0.348      0.144          0.136  0.217       -0.024   0.071
RV_weekly         0.564      1.000       0.655      0.262          0.219  0.370       -0.039   0.137
RV_monthly        0.348      0.655       1.000      0.401          0.138  0.435       -0.060   0.206
CDS_level         0.144      0.262       0.401      1.000          0.070  0.169        0.049   0.165
CDS_change_5d     0.136      0.219       0.138      0.070          1.000  0.117       -0.001   0.013
VIX               0.217      0.370       0.435      0.169          0.117  1.000       -0.023   0.355
Yield_slope      -0.024     -0.039      -0.060      0.049         -0.001 -0.023        1.000   0.554
BAA10Y            0.071      0.137       0.206      0.165          0.013  0.355        0.554   1.000


In [3]:
# Intercept column excluded from VIFs.
X = np.column_stack([np.ones(len(d)), d[HARX].values])
vif = pd.Series(
    {HARX[i - 1]: variance_inflation_factor(X, i) for i in range(1, X.shape[1])}
).round(2)
vif.to_csv("VIF.csv")
print("Variance inflation factors:\n")
print(vif.to_string())

Variance inflation factors:

RV_daily         1.47
RV_weekly        2.33
RV_monthly       2.12
CDS_level        1.20
CDS_change_5d    1.05
VIX              1.46
Yield_slope      1.59
BAA10Y           1.83


In [4]:
# Test each firm's residuals for autocorrelation at lag 10.
# Firms with fewer than 60 observations are skipped.
ljung = {}
for name, cols in [("HAR", HAR), ("HARX", HARX)]:
    m = LinearRegression().fit(d[cols], d["Target_h1"])
    resid = d["Target_h1"] - m.predict(d[cols])
    resid_df = d.assign(_resid=resid)
    rejecting = tested = 0
    for firm, g in resid_df.groupby("Firm"):
        if len(g) < 60:
            continue
        tested += 1
        pval = acorr_ljungbox(g.sort_values("Date")["_resid"].values,
                              lags=[10]).iloc[0]["lb_pvalue"]
        rejecting += (pval < 0.05)
    ljung[name] = {"rejecting": rejecting, "tested": tested}
ljung_df = pd.DataFrame(ljung).T
ljung_df.to_csv("Ljung_Box.csv")
print("Ljung-Box test (lag 10) on HAR residuals, firms rejecting at 5%:\n")
print(ljung_df.to_string())

Ljung-Box test (lag 10) on HAR residuals, firms rejecting at 5%:

      rejecting  tested
HAR          54      55
HARX         55      55
